# NER with BERT + Global Pointer

Span-based NER (Su Jianlin, 2022). For each entity type, the model predicts a score over every (start, end) span instead of per-token BIO tags. Naturally handles nested/overlapping entities. Loss is multi-label categorical CE.

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers seqeval

In [ ]:
# Cell 2: Imports
import os
import sys
import math
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from collections import Counter, defaultdict

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Cell 3: Config
# Defaults are tuned for Colab T4. On CPU drop BATCH_SIZE to 4 and EPOCHS to 3.
MODEL_NAME = 'bert-base-cased'
MAX_LEN    = 128
BATCH_SIZE = 16 if torch.cuda.is_available() else 4
EPOCHS     = 10 if torch.cuda.is_available() else 3
LR         = 3e-5
HEAD_SIZE  = 64        # per-entity inner dimension for q/k
USE_ROPE   = True

In [ ]:
# Cell 4: Locate and read CoNLL files
# Works on Colab and locally without manual path editing.
#   Local: walks up from cwd to find data/processed/ in the project tree.
#   Colab: tries /content/data/processed, /content/drive/MyDrive/NER/data/processed,
#          then /content/ (bare files) - does NOT auto-mount Drive.
IN_COLAB = 'google.colab' in sys.modules

def resolve_data_dir():
    candidates = []
    if IN_COLAB:
        candidates += [
            '/content/data/processed',
            '/content/drive/MyDrive/NER/data/processed',
            '/content',
        ]
    cur = os.getcwd()
    for _ in range(6):
        candidates.append(os.path.join(cur, 'data', 'processed'))
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent

    for d in candidates:
        if os.path.exists(os.path.join(d, 'train.conll')):
            return os.path.abspath(d)

    msg = ['Could not find train.conll. Tried:'] + ['  ' + os.path.abspath(c) for c in candidates]
    if IN_COLAB:
        msg += [
            '',
            'On Colab, upload the three .conll files into /content/ and re-run this cell:',
            '    from google.colab import files',
            '    files.upload()  # select train.conll, valid.conll, test.conll',
        ]
    raise FileNotFoundError('\n'.join(msg))

def read_conll(filepath):
    """Read CoNLL file; return list of (tokens, tags) per sentence."""
    sentences, tokens, tags = [], [], []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if not line.strip():
                if tokens:
                    sentences.append((tokens, tags))
                    tokens, tags = [], []
            else:
                parts = line.split('\t')
                if len(parts) == 2:
                    tokens.append(parts[0])
                    tags.append(parts[1])
    if tokens:
        sentences.append((tokens, tags))
    return sentences

DATA_DIR = resolve_data_dir()
print(f'DATA_DIR: {DATA_DIR}')

train_data = read_conll(os.path.join(DATA_DIR, 'train.conll'))
valid_data = read_conll(os.path.join(DATA_DIR, 'valid.conll'))
test_data  = read_conll(os.path.join(DATA_DIR, 'test.conll'))

print(f'Train: {len(train_data)} docs')
print(f'Valid: {len(valid_data)} docs')
print(f'Test:  {len(test_data)} docs')

In [ ]:
# Cell 5: BIO tags -> entity spans, build entity-type vocab
def bio_to_spans(tags):
    """Return list of (word_start, word_end_inclusive, entity_type)."""
    spans, i = [], 0
    while i < len(tags):
        t = tags[i]
        if t.startswith('B-'):
            ent = t[2:]
            j = i + 1
            while j < len(tags) and tags[j] == 'I-' + ent:
                j += 1
            spans.append((i, j - 1, ent))
            i = j
        elif t.startswith('I-'):
            # orphan I- (no preceding B-): treat as a B- of same type
            ent = t[2:]
            j = i + 1
            while j < len(tags) and tags[j] == 'I-' + ent:
                j += 1
            spans.append((i, j - 1, ent))
            i = j
        else:
            i += 1
    return spans

entity_types = set()
for tokens, tags in train_data + valid_data + test_data:
    for _, _, ent in bio_to_spans(tags):
        entity_types.add(ent)

entity_list = sorted(entity_types)
ent2id = {e: i for i, e in enumerate(entity_list)}
id2ent = {i: e for e, i in ent2id.items()}
NUM_ENT = len(entity_list)

print(f'Entity types ({NUM_ENT}): {entity_list}')
span_counts = Counter(ent for _, tags in train_data for _, _, ent in bio_to_spans(tags))
print(f'Train span counts: {dict(span_counts)}')

In [ ]:
# Cell 6: Dataset
# Tokenize word-by-word, record each word's subword span, then map gold word
# spans into subword spans. We keep sub2word so predicted subword spans can be
# mapped back to word-level for evaluation.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class GPDataset(Dataset):
    def __init__(self, data, tokenizer, ent2id, max_len):
        self.data = data
        self.tok = tokenizer
        self.ent2id = ent2id
        self.max_len = max_len
        self.num_ent = len(ent2id)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, tags = self.data[idx]
        word_spans = bio_to_spans(tags)

        input_ids = [self.tok.cls_token_id]
        word_to_sub = []   # word_to_sub[i] = (sub_start, sub_end_inclusive)
        for word in tokens:
            sub = self.tok.encode(word, add_special_tokens=False)
            if not sub:
                # mark as missing; spans referencing this word will be dropped
                word_to_sub.append(None)
                continue
            if len(input_ids) + len(sub) > self.max_len - 1:
                word_to_sub.append(None)
                continue
            start = len(input_ids)
            input_ids.extend(sub)
            word_to_sub.append((start, len(input_ids) - 1))
        input_ids.append(self.tok.sep_token_id)

        seq_len = len(input_ids)
        pad_len = self.max_len - seq_len
        attention_mask = [1] * seq_len + [0] * pad_len
        input_ids += [self.tok.pad_token_id] * pad_len

        # build [num_ent, max_len, max_len] target tensor
        labels = torch.zeros(self.num_ent, self.max_len, self.max_len, dtype=torch.float)
        gold_word_spans = []
        for ws, we, ent in word_spans:
            if we >= len(word_to_sub):
                continue
            ms = word_to_sub[ws]
            me = word_to_sub[we]
            if ms is None or me is None:
                continue
            ss, se = ms[0], me[1]
            labels[self.ent2id[ent], ss, se] = 1.0
            gold_word_spans.append((ws, we, ent))

        # sub2word for decoding back to word indices
        sub2word = [-1] * self.max_len
        for w_idx, span in enumerate(word_to_sub):
            if span is None:
                continue
            for k in range(span[0], span[1] + 1):
                sub2word[k] = w_idx

        return {
            'input_ids':      torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'labels':         labels,
            'sub2word':       torch.tensor(sub2word, dtype=torch.long),
            'gold_spans':     gold_word_spans,
            'seq_len':        seq_len,
        }

def collate(batch):
    out = {
        'input_ids':      torch.stack([b['input_ids'] for b in batch]),
        'attention_mask': torch.stack([b['attention_mask'] for b in batch]),
        'labels':         torch.stack([b['labels'] for b in batch]),
        'sub2word':       torch.stack([b['sub2word'] for b in batch]),
        'seq_len':        torch.tensor([b['seq_len'] for b in batch]),
        'gold_spans':     [b['gold_spans'] for b in batch],
    }
    return out

train_ds = GPDataset(train_data, tokenizer, ent2id, MAX_LEN)
valid_ds = GPDataset(valid_data, tokenizer, ent2id, MAX_LEN)
test_ds  = GPDataset(test_data,  tokenizer, ent2id, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, collate_fn=collate)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, collate_fn=collate)

print(f'Train batches: {len(train_loader)}, Valid: {len(valid_loader)}, Test: {len(test_loader)}')

In [ ]:
# Cell 7: Global Pointer head + model
# For each entity type, two linear projections q, k of size HEAD_SIZE.
# logit[b, ent, i, j] = q_b,i,ent . k_b,j,ent  (with optional RoPE on q and k).
# Padding and i > j positions are masked with -1e12.
class GlobalPointer(nn.Module):
    def __init__(self, hidden_size, num_ent, head_size, use_rope=True):
        super().__init__()
        self.num_ent = num_ent
        self.head_size = head_size
        self.use_rope = use_rope
        self.dense = nn.Linear(hidden_size, num_ent * head_size * 2)

    @staticmethod
    def _rope_emb(seq_len, dim, device):
        pos = torch.arange(seq_len, dtype=torch.float, device=device).unsqueeze(-1)
        idx = torch.arange(dim // 2, dtype=torch.float, device=device)
        freq = torch.pow(10000.0, -2 * idx / dim)
        emb = pos * freq                                        # [L, dim/2]
        sin, cos = emb.sin(), emb.cos()
        sin = sin.repeat_interleave(2, dim=-1)                  # [L, dim]
        cos = cos.repeat_interleave(2, dim=-1)
        return sin, cos

    def _apply_rope(self, x, sin, cos):
        # x: [B, L, E, H]; sin/cos: [L, H]
        x2 = torch.stack([-x[..., 1::2], x[..., ::2]], dim=-1).reshape_as(x)
        sin = sin.view(1, -1, 1, x.size(-1))
        cos = cos.view(1, -1, 1, x.size(-1))
        return x * cos + x2 * sin

    def forward(self, hidden, attention_mask):
        B, L, _ = hidden.size()
        x = self.dense(hidden)                                  # [B, L, E*2H]
        x = x.view(B, L, self.num_ent, 2 * self.head_size)
        q = x[..., :self.head_size]
        k = x[..., self.head_size:]

        if self.use_rope:
            sin, cos = self._rope_emb(L, self.head_size, hidden.device)
            q = self._apply_rope(q, sin, cos)
            k = self._apply_rope(k, sin, cos)

        # logits[b, e, i, j] = sum_h q[b,i,e,h] * k[b,j,e,h]
        logits = torch.einsum('bieh,bjeh->beij', q, k) / math.sqrt(self.head_size)

        # mask padding positions on both axes
        m = attention_mask.float()
        pad_mask = m.unsqueeze(1) * m.unsqueeze(2)              # [B, L, L]
        logits = logits - (1.0 - pad_mask).unsqueeze(1) * 1e12
        # mask i > j (start must be <= end)
        tri = torch.tril(torch.ones(L, L, device=hidden.device), diagonal=-1)
        logits = logits - tri.view(1, 1, L, L) * 1e12
        return logits


class BertGlobalPointer(nn.Module):
    def __init__(self, model_name, num_ent, head_size, use_rope):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.gp = GlobalPointer(self.bert.config.hidden_size, num_ent, head_size, use_rope)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        return self.gp(out.last_hidden_state, attention_mask)


model = BertGlobalPointer(MODEL_NAME, NUM_ENT, HEAD_SIZE, USE_ROPE).to(device)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Cell 8: Multi-label categorical CE loss (Su Jianlin).
# Treat each (entity, i, j) cell as an independent binary slot, then apply
# multi-label CE over the flattened E*L*L set per sample.
def multilabel_ce(y_pred, y_true):
    # both [B, E, L, L]
    B = y_pred.size(0)
    y_pred = y_pred.reshape(B, -1)
    y_true = y_true.reshape(B, -1)
    y_pred = (1 - 2 * y_true) * y_pred
    y_pred_neg = y_pred - y_true * 1e12
    y_pred_pos = y_pred - (1 - y_true) * 1e12
    zeros = torch.zeros_like(y_pred[..., :1])
    y_pred_neg = torch.cat([y_pred_neg, zeros], dim=-1)
    y_pred_pos = torch.cat([y_pred_pos, zeros], dim=-1)
    neg = torch.logsumexp(y_pred_neg, dim=-1)
    pos = torch.logsumexp(y_pred_pos, dim=-1)
    return (neg + pos).mean()

In [ ]:
# Cell 9: Optimizer & scheduler
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
)

In [ ]:
# Cell 10: Evaluation - decode subword spans, map back to word level, compute F1
def decode_batch(logits, sub2word, threshold=0.0):
    """logits: [B, E, L, L]. Returns list[B] of sets of (word_start, word_end, ent)."""
    out = []
    B = logits.size(0)
    for b in range(B):
        spans = set()
        ent_idx, i_idx, j_idx = torch.where(logits[b] > threshold)
        s2w = sub2word[b].tolist()
        for e, i, j in zip(ent_idx.tolist(), i_idx.tolist(), j_idx.tolist()):
            ws, we = s2w[i], s2w[j]
            if ws < 0 or we < 0 or ws > we:
                continue
            spans.add((ws, we, id2ent[e]))
        out.append(spans)
    return out

def evaluate(model, dataloader):
    model.eval()
    tp = fp = fn = 0
    per_ent = defaultdict(lambda: [0, 0, 0])  # tp, fp, fn
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attn = batch['attention_mask'].to(device)
            sub2word = batch['sub2word']
            logits = model(input_ids, attn).cpu()
            preds = decode_batch(logits, sub2word)
            for pred, gold in zip(preds, batch['gold_spans']):
                gold = set(gold)
                tp += len(pred & gold)
                fp += len(pred - gold)
                fn += len(gold - pred)
                for s in pred & gold: per_ent[s[2]][0] += 1
                for s in pred - gold: per_ent[s[2]][1] += 1
                for s in gold - pred: per_ent[s[2]][2] += 1
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f, per_ent

In [ ]:
# Cell 11: Training loop
best_f1 = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for step, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attn      = batch['attention_mask'].to(device)
        labels    = batch['labels'].to(device)

        logits = model(input_ids, attn)
        loss = multilabel_ce(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()

        if (step + 1) % 50 == 0:
            print(f'  Epoch {epoch} Step {step+1}/{len(train_loader)} Loss: {loss.item():.4f}')

    avg_loss = total_loss / max(1, len(train_loader))
    p, r, f1, _ = evaluate(model, valid_loader)
    print(f'Epoch {epoch}/{EPOCHS} - Loss: {avg_loss:.4f} | Val P: {p:.4f} R: {r:.4f} F1: {f1:.4f}')

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), 'best_global_pointer.pt')
        print(f'  -> Saved best model (F1: {f1:.4f})')

print(f'\nTraining complete. Best Val F1: {best_f1:.4f}')

In [ ]:
# Cell 12: Test evaluation
model.load_state_dict(torch.load('best_global_pointer.pt'))
p, r, f1, per_ent = evaluate(model, test_loader)

print('=' * 60)
print('TEST RESULTS - BERT + Global Pointer')
print('=' * 60)
print(f'Precision: {p:.4f}')
print(f'Recall:    {r:.4f}')
print(f'F1-Score:  {f1:.4f}')
print('=' * 60)
print()
print(f'{"entity":<12}{"precision":>11}{"recall":>11}{"f1":>11}{"support":>11}')
for ent in sorted(per_ent):
    tp, fp, fn = per_ent[ent]
    pe = tp / (tp + fp) if tp + fp else 0.0
    re = tp / (tp + fn) if tp + fn else 0.0
    fe = 2 * pe * re / (pe + re) if pe + re else 0.0
    print(f'{ent:<12}{pe:>11.4f}{re:>11.4f}{fe:>11.4f}{tp+fn:>11d}')

In [ ]:
# Cell 13 (Optional): Save model for deployment
import json

os.makedirs('saved_model_global_pointer', exist_ok=True)
torch.save(model.state_dict(), 'saved_model_global_pointer/model.pt')
with open('saved_model_global_pointer/labels.json', 'w') as f:
    json.dump({
        'ent2id': ent2id,
        'id2ent': {str(k): v for k, v in id2ent.items()},
        'head_size': HEAD_SIZE,
        'use_rope': USE_ROPE,
        'max_len': MAX_LEN,
        'model_name': MODEL_NAME,
    }, f)
tokenizer.save_pretrained('saved_model_global_pointer')
print('Model saved to saved_model_global_pointer/')